# Aravis BlackFly Camera Driver — Hardware Test

Tests the Aravis driver on real Cytation hardware. Two parts:

- **Part 1** (Steps 1–9b): Camera only via raw Aravis. No serial, no motors, safe.
- **Part 2** (Steps 10–13): Full Cytation integration via `CytationAravisBackend`.

## Prerequisites

```bash
# 1. Aravis system library
brew install aravis          # macOS

# 2. Python dependencies
cd ~/vs\ code/pylabrobot
poetry install

# 3. Register Jupyter kernel
poetry run python -m ipykernel install --user --name pylabrobot-workspace --display-name "PyLabRobot (Aravis)"
```

Then select **PyLabRobot (Aravis)** as kernel in VS Code (top-right).

---

# Part 1: Camera Only (Aravis Direct)

## Step 0: Verify Kernel and Dependencies

In [ ]:
import sys

print(f"Python: {sys.executable}")
print(f"Version: {sys.version}\n")

deps = {"numpy": "numpy", "matplotlib": "matplotlib", "pylibftdi": "pylibftdi", "pylabrobot": "pylabrobot"}
all_ok = True
for name, module in deps.items():
    try:
        __import__(module)
        print(f"  {name}: OK")
    except ImportError:
        print(f"  {name}: MISSING")
        all_ok = False

try:
    import gi
    gi.require_version("Aravis", "0.8")
    from gi.repository import Aravis
    print(f"  aravis: OK")
except (ImportError, ValueError):
    print(f"  aravis: MISSING — brew install aravis && pip install PyGObject")
    all_ok = False

print(f"\n{'All checks passed!' if all_ok else 'Fix missing dependencies before continuing.'}")

## Step 1: Verify Aravis Installation

In [ ]:
import subprocess
import gi

gi.require_version("Aravis", "0.8")
from gi.repository import Aravis


def _aravis_library_version():
    """Aravis C library semver; not exposed on the gi.repository.Aravis module."""
    for pkg in ("aravis-0.8", "aravis"):
        try:
            out = subprocess.check_output(
                ["pkg-config", "--modversion", pkg],
                stderr=subprocess.DEVNULL,
                text=True,
            )
            return out.strip()
        except (subprocess.CalledProcessError, FileNotFoundError):
            continue
    return None


print("Aravis loaded successfully")
lib_ver = _aravis_library_version()
if lib_ver:
    print(f"Aravis library version: {lib_ver}")
else:
    print("Aravis library version: (unknown — pkg-config not available)")
print(f"GObject introspection API: Aravis {Aravis._version}")

## Step 2: Discover Cameras

In [ ]:
Aravis.update_device_list()
n_devices = Aravis.get_n_devices()
print(f"Found {n_devices} camera(s):\n")

for i in range(n_devices):
    device_id = Aravis.get_device_id(i)
    print(f"  [{i}] {device_id}")

## Step 3: Connect to the BlackFly

Set `SERIAL` to your camera's serial number from Step 2, or `None` for first camera.

In [ ]:
SERIAL = None  # e.g., "17525520"

camera = Aravis.Camera.new(SERIAL)
device = camera.get_device()

print(f"Connected!")
print(f"  Model:    {device.get_string_feature_value('DeviceModelName')}")
print(f"  Serial:   {device.get_string_feature_value('DeviceSerialNumber')}")
print(f"  Vendor:   {device.get_string_feature_value('DeviceVendorName')}")
print(f"  Firmware: {device.get_string_feature_value('DeviceFirmwareVersion')}")

## Step 4: Read Camera Properties

In [ ]:
x, y, width, height = camera.get_region()
payload = camera.get_payload()

print(f"Image region: {width} x {height} (offset: {x}, {y})")
print(f"Payload size: {payload} bytes")
print(f"Pixel format: {camera.get_pixel_format_as_string()}")
print(f"")
print(f"Exposure range: {camera.get_exposure_time_bounds()} \u00b5s")
print(f"Current exposure: {camera.get_exposure_time()} \u00b5s ({camera.get_exposure_time()/1000:.1f} ms)")
print(f"")
print(f"Gain range: {camera.get_gain_bounds()}")
print(f"Current gain: {camera.get_gain()}")

## Step 5: Configure Software Trigger

In [ ]:
device.set_string_feature_value("TriggerSelector", "FrameStart")
device.set_string_feature_value("TriggerSource", "Software")
device.set_string_feature_value("TriggerMode", "On")

print("Software trigger configured:")
print(f"  TriggerSelector: {device.get_string_feature_value('TriggerSelector')}")
print(f"  TriggerSource:   {device.get_string_feature_value('TriggerSource')}")
print(f"  TriggerMode:     {device.get_string_feature_value('TriggerMode')}")

## Step 6: Set Exposure and Gain

In [ ]:
EXPOSURE_MS = 10.0
GAIN = 1.0

device.set_string_feature_value("ExposureAuto", "Off")
camera.set_exposure_time(EXPOSURE_MS * 1000)
print(f"Exposure set to {camera.get_exposure_time()} \u00b5s ({camera.get_exposure_time()/1000:.1f} ms)")

device.set_string_feature_value("GainAuto", "Off")
camera.set_gain(GAIN)
print(f"Gain set to {camera.get_gain()}")

## Step 7: Capture a Single Frame

If this cell works, **Aravis can control your BlackFly**.

In [ ]:
import time
import numpy as np

# BlackFly needs a delay after trigger mode change (same as Rick's CytationBackend)
time.sleep(1)

# Allocate stream and buffers
stream = camera.create_stream(None, None)
payload = camera.get_payload()
for _ in range(5):
    stream.push_buffer(Aravis.Buffer.new_allocate(payload))

# Start acquisition, software trigger, grab buffer
camera.start_acquisition()
device.execute_command("TriggerSoftware")
buffer = stream.timeout_pop_buffer(10_000_000)

if buffer is None:
    print("ERROR: Capture timed out!")
    print(f"  Pixel format: {camera.get_pixel_format_as_string()}")
    print(f"  Payload: {payload}, Region: {camera.get_region()}")
else:
    data = buffer.get_data()
    _, _, width, height = camera.get_region()
    image = np.frombuffer(data, dtype=np.uint8).reshape(height, width).copy()
    stream.push_buffer(buffer)
    print(f"Captured! Shape: {image.shape}, dtype: {image.dtype}")
    print(f"  Min: {image.min()}, Max: {image.max()}, Mean: {image.mean():.1f}")

camera.stop_acquisition()

## Step 8: Display the Image

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.imshow(image, cmap='gray', vmin=0, vmax=255)
ax1.set_title(f'BlackFly via Aravis \u2014 {EXPOSURE_MS}ms, gain={GAIN}')
ax2.hist(image.ravel(), bins=256, range=(0, 255), color='gray', alpha=0.7)
ax2.axvline(image.mean(), color='red', linestyle='--', label=f'mean={image.mean():.0f}')
ax2.set_title('Pixel intensity histogram')
ax2.legend()
plt.tight_layout()
plt.show()

## Step 9b: Disconnect Raw Camera

**Run this before Part 2.** Releases the camera so `AravisCamera`/`CytationAravisBackend` can reconnect.

In [ ]:
import gc, time

try:
    camera.stop_acquisition()
except Exception:
    pass
try:
    device.set_string_feature_value("TriggerMode", "Off")
except Exception:
    pass

stream = None
device = None
camera = None

gc.collect()
gc.collect()
time.sleep(2)

Aravis.update_device_list()
n = Aravis.get_n_devices()
print(f"Camera released. {n} device(s) available." if n > 0 else "WARNING: Camera not released. Restart kernel.")

---

# Part 2: Cytation Integration (PLR Backend)

Uses `CytationAravisBackend` — Aravis for the camera, PLR's `BioTekBackend` for the Cytation serial protocol (filter wheel, objectives, focus, LED, stage).

**Requires**: Cytation powered on + BlackFly connected via USB.

## Step 10: Setup CytationAravisBackend

Equivalent of Rick's `CytationBackend()` setup — no Spinnaker env var needed.

In [ ]:
import matplotlib.pyplot as plt
from pylabrobot.capabilities.microscopy.standard import ImagingMode, Objective
from pylabrobot.agilent.biotek.cytation_aravis import CytationAravisBackend

CAMERA_SERIAL = "010B6B10"  # from Step 2

backend = CytationAravisBackend(camera_serial=CAMERA_SERIAL)
await backend.setup(use_cam=True)

print("CytationAravisBackend ready")
print(f"  Firmware: {backend.version}")
print(f"  Objectives: {backend.objectives}")
print(f"  Filters: {backend.filters}")

## Step 11: Open Door, Load Plate, Close Door

In [ ]:
await backend.open(slow=False)
print("Door open \u2014 insert plate, then run next cell")

In [ ]:
await backend.close(slow=True)
print("Door closed")

## Step 12: Single Capture

Adjust `objective`, `focal_height`, `exposure_time`, `gain` for your plate and objectives.

In [ ]:
from pylabrobot.resources import CellVis_24_wellplate_3600uL_Fb
plate = CellVis_24_wellplate_3600uL_Fb(name="plate")

res = await backend.capture(
    row=2,
    column=3,
    mode=ImagingMode.BRIGHTFIELD,
    objective=Objective.O_2_5X_FL_Zeiss,  # your objectives: O_4X_PL_FL, O_2_5X_FL_Zeiss
    focal_height=3,
    exposure_time=5,
    gain=16,
    plate=plate,
    led_intensity=5,
)

plt.figure(figsize=(8, 6))
plt.imshow(res.images[0], cmap="gray", vmin=0, vmax=255)
plt.title(f"Brightfield 2.5x \u2014 {res.exposure_time:.1f} ms, focal={res.focal_height:.3f} mm")
plt.colorbar(label="pixel value")
plt.show()
print(f"Shape: {res.images[0].shape}, Mean: {res.images[0].mean():.1f}")

## Step 13: Multi-Image Coverage (Tiling)

In [ ]:
num_rows = 4
num_cols = 3

res = await backend.capture(
    row=2,
    column=3,
    mode=ImagingMode.BRIGHTFIELD,
    objective=Objective.O_2_5X_FL_Zeiss,
    focal_height=3,
    exposure_time=5,
    gain=16,
    plate=plate,
    led_intensity=5,
    coverage=(num_rows, num_cols),
    center_position=(0, 0),
)

print(f"Captured {len(res.images)} images")

fig = plt.figure(figsize=(12, 8))
for row in range(num_rows):
    for col in range(num_cols):
        plt.subplot(num_rows, num_cols, row * num_cols + col + 1)
        plt.imshow(res.images[row * num_cols + col], cmap="gray", vmin=0, vmax=255)
        plt.axis("off")
plt.suptitle(f"Brightfield 2.5x Zeiss \u2014 {num_rows}x{num_cols} coverage")
plt.tight_layout()
plt.show()

## Cleanup

Always run this when done.

In [ ]:
await backend.stop()
print("Backend stopped \u2014 camera and serial released")